In [ ]:
import time

%matplotlib widget
import matplotlib.pyplot as plt

import pennylane as qml
from pennylane import numpy as np

In [ ]:
infile = "traf_runs.csv"
outdir = "tmp"
verbose = True

# Circuit parameters
n_qubits = 6
layers = 6
global_rots = 3
enc_layers = [0]

device = qml.device("lightning.qubit", wires=n_qubits)

options = {"maxiter": 100, "term": 1e-4}

In [ ]:
# Circuit design
if global_rots not in [2, 3]:
    raise Exception(f"global rotations must be 2 or 3, got {global_rots}")

def rotations(wire, params):
    qml.RZ(params[0], wires=wire)
    qml.RY(params[1], wires=wire)
    if global_rots == 3:
        qml.RZ(params[2], wires=wire)


def entangle(n_qubits):
    if n_qubits <= 1:
        return
    for ii in range(n_qubits):
        qml.CNOT(wires=[ii, (ii+1) % n_qubits])


def training_layer(n_qubits, params):
    for ii in range(n_qubits):
        rotations(ii, params[ii,:])
    entangle(n_qubits)


def encoding_layer(n_qubits, params):
    for ii in range(n_qubits):
        qml.RX(params[ii], wires=ii)


def enc_params(n_qubits):
    params = [np.random.uniform(-2*np.pi, np.pi) for _ in range(n_qubits)]
    return params

def rot_params(n_qubits, layers):
    params = np.empty((layers, n_qubits, global_rots), np.float64)
    for ll in range(layers):
        for nn in range(n_qubits):
            for aa in range(global_rots):
                params[ll, nn, aa] = np.random.uniform(-2*np.pi, np.pi)
    return params


@qml.qnode(device, diff_method="adjoint")
def circuit(n_qubits, layers, enc_params, rot_params, enc_layers):        
    for ii in range(layers):
        if ii in enc_layers:
            encoding_layer(n_qubits, enc_params)
        training_layer(n_qubits, rot_params[ii])
    exp = qml.PauliZ(0)
    for ii in range(1, n_qubits):
        exp = exp @ qml.PauliZ(ii)
    return qml.expval(exp)

In [ ]:
# Process target
def expectation(probs):
    # use expectation value to predict efficiency
    val = 0
    for key, value in probs.items():
        eig = 1
        for char in key:
            if char == "1":
                eig *= -1
        val += eig * value
    return val

def linmap(value, amin, amax, bmin, bmax):
    return bmin + (bmax - bmin)/(amax - amin) * (value - amin)

def polymap(value, amin, amax, bmin, bmax, power=2):
    m = linmap(value, amin, amax, 0, 1)
    m = m**power
    return linmap(m, 0, 1, bmin, bmax)

def expmap(value, amin, amax, bmin, bmax, base=np.exp(1)):
    m = linmap(value, amin, amax, 0, 1)
    m = base**m
    return linmap(m, 1, base, bmin, bmax)

def map_expval(expval, start, stop):
    return linmap(expval, -1, 1, start, stop)

data = np.loadtxt(infile, delimiter=",", usecols=(3,4,5,6,7), skiprows = 1)
x = range(0, data.shape[0])
s = (len(x), 6)
target = np.zeros(s)

In [ ]:
# target mapping
data[x, 4] = 1 - data[x, 4]
adelta = 1
tdelta = 0.3
amin = np.min(data[x,0:4])
amax = np.max(data[x,0:4])
tmin = np.min(data[x,4])
tmax = np.max(data[x,4])

target[:,0:4] = linmap(data[x,0:4], amin, amax, 0 + adelta, 2*np.pi - adelta)
target[:,4] = polymap(data[x,4], tmin, tmax, -1 + tdelta, 1 - tdelta, 50)
target[:,5] = linmap(data[x,4], tmin, tmax, -1 + tdelta, 1 - tdelta)

In [ ]:
plt.close()
plt.plot(target[:,0], target[:,4])

In [ ]:
def new_tracker():
    tracker = {
        "count": 0,  # Elapsed optimization steps
        "cost": [],  # Cost at each step
        "params": [],  # Track parameters
        "time": [], # Step time
    }
    return tracker

In [ ]:
# pennylane optimisers
def train_opt(opt, params, target, circuit, n_qubits, layers, enc_layers, options, tracker, verbose=True):
    """Function to train VQE"""
    print("Starting the training.")

    print("=" * 80)
    print(f"OPTIMIZATION on cpu for {n_qubits} qubits, {layers} layers, {global_rots} rotations per set")
    print(f"Encoding layers {enc_layers}")
    if "maxiter" in options:
        print("Maximum iterations:", options["maxiter"])
    if "term" in options:
        print("Termination condition:", options["term"])

    print("Circuit configuration")
    print(qml.draw(circuit, max_length=140)(3, 2, [0, 0, 0], np.zeros((2,3,global_rots)), enc_layers))

    if not verbose:
        print('Param "verbose" set to False. Will not print intermediate steps.')
        print("=" * 80)

    def objective(params):
        angles = target[:,0]
        angles = np.tile(angles, (n_qubits,1))
        a = np.clip(circuit(n_qubits, layers, angles, params, enc_layers), -0.5, np.inf)
        b = np.clip(target[:,1], -0.5, np.inf)
        return np.mean(np.square(a - b))

    try:
        for i in range(options["maxiter"]):
            tracker.update({"count": tracker["count"] + 1})
            if verbose:
                print("=" * 80)
                print("Iteration step. Cycle:", tracker["count"])

            t1 = time.time()
            params, cost = opt.step_and_cost(objective, params)
            t2 = time.time()

            cost_delta = tracker["cost"][-1] - cost if i > 0 else 1

            if verbose:
                print("cost:", cost)
                print("cost delta:", cost_delta)
                print("time:", t2-t1)

            # update tracker
            tracker["cost"].append(cost)
            tracker["params"].append(params)
            tracker["time"].append(t2-t1)

            # termination condition
            if "term" in options and options["term"] > cost:
                print("Termination condition reached at step:", i)
                break
    except KeyboardInterrupt:
        print("Keyboard interrupt, stopping training")

    # final run
    tracker["cost"].append(objective(params))
    tracker["params"].append(params)

    minid = np.argmin(tracker["cost"], requires_grad=False)
    cost = tracker["cost"][minid]
    params = tracker["params"][minid]
    print("Final cost:", cost)
    print("Final angles:", params)
    print("Training complete.")

    return cost, params, tracker

In [ ]:
init_params = rot_params(n_qubits, layers)

In [ ]:
qml.draw_mpl(circuit)(n_qubits, layers, enc_params(n_qubits), init_params, enc_layers)

In [ ]:
# run circuit with final parameters
angles = np.linspace(0, 2*np.pi, 100)
angles = np.tile(angles, (n_qubits,1))
fparam1 = np.load("tmp/q6l6r3_adam.npy")
out1 = circuit(n_qubits, layers, angles, fparam1, [0])

plt.figure()
#plt.plot(angles[0,:], out)
plt.plot(angles[0,:], out1)
plt.plot(target[:,0], target[:,1])
plt.xlabel("input angle")
plt.ylabel("output expectation value")